# 05 2026 인력 배치 권고표 (2026-01-01 ~ 01-28, 4주)

C3(Chronos-2 + 공휴일·무료 개방)로 2025-12-31 까지의 기록에서 28일을 한 번에 예측. 인력(소수) = q0.8 ÷ 2,000, 휴관일 0.
**정수 = 날마다 올림** — 예측이 불확실할 때 「충분할 확률 ≥ 0.8 인 가장 작은 정수」가 q0.8 의 올림이다.
k주차는 (k − 1)주 일찍 확정한 백테스트(lead 1 · 8 · 15 · 22, 04 노트북 2절)로 검증했다 — C3 비용 8.46 · 9.22 · 9.44 · 9.30억, 적중률 0.80 유지.

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import font_manager

# 묶음 폴더(real_final 또는 final) 찾기 — notebooks/ 에서 열든, 묶음 폴더나 그 위(Colab 의 /content)에서 열든 된다
HERE = Path.cwd()
CANDIDATES = [p / sub for p in [HERE, *HERE.parents] for sub in ('data', 'real_final/data', 'final/data')]
DATA = next((c for c in CANDIDATES if (c / 'metrics.csv').exists()), None)
if DATA is None:
    raise FileNotFoundError(f'data/metrics.csv 를 찾지 못했습니다 (지금 위치: {HERE}). '
                            '묶음 폴더 전체(data · notebooks ...)를 받았는지, 노트북을 notebooks/ 에서 열었는지 확인하세요.')
BASE = DATA.parent                              # 묶음 폴더
# 01 노트북만 쓰는 src/ 와 원본 데이터: 묶음 폴더 안(real_final: src/ · raw/)이 먼저, 없으면 저장소 최상위
SRC = next((p for p in [BASE / 'src', BASE.parent / 'src'] if (p / 'hv.py').exists()), None)
RAW = next((p for p in [BASE / 'raw', BASE.parent] if (p / 'anchors.json').exists()), None)
if SRC is not None:
    sys.path.insert(0, str(SRC))
print('데이터:', DATA)

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 80)
HAVE = {f.name for f in font_manager.fontManager.ttflist}
KFONT = next((f for f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Noto Sans KR'] if f in HAVE), None)
if KFONT is None:
    print('※ 한글 폰트가 없어 그림의 한글이 네모로 보일 수 있습니다 (Colab: !apt -qq install fonts-nanum 후 런타임 다시 시작)')
plt.rcParams.update({'font.family': KFONT or 'sans-serif', 'axes.unicode_minus': False, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
SITES = ['경복궁', '창덕궁', '창경궁', '덕수궁', '종묘', '동구릉', '광릉', '홍유릉', '사릉', '태릉', '정릉', '헌릉', '선릉', '의릉',
         '영휘원', '서오릉', '서삼릉', '융릉', '파주삼릉', '파주장릉', '김포장릉', '양주온릉', '세종대왕유적']   # 궁 → 종묘 → 왕릉
BLUE, ORANGE, AQUA, GRAY, VIOLET = '#2a78d6', '#eb6834', '#1baf7a', '#898781', '#6250d6'

M = pd.read_csv(DATA / 'metrics.csv')
ORDER = M.drop_duplicates('방식').sort_values('방식 순서')['방식'].tolist()

def table(axis, metric, conf='전날', models=None, scale=1):
    """축(axis)의 값별로 방식 × 값 표. scale=1e8 이면 억 원."""
    x = M[(M['축'] == axis) & (M['확정'] == conf)]
    t = x.pivot(index='방식', columns='값', values=metric) / scale
    idx = [m for m in ORDER if m in t.index and (models is None or m in models)]
    return t.reindex(idx)

KEY = ['1 직전 7일 평균', '2 + 서비스 수준 보정', '7 MLP + 마지막 수요일 (FVA 2차)', 'C3 Chronos-2 + 공휴일·무료 개방']

데이터: C:\Users\USER\Desktop\real_final\data


In [2]:
R = pd.read_csv(DATA / 'recommend_2026.csv', parse_dates=['date'])
print('무료 개방일:', list(R.loc[R['is_free_rule'] == 1, 'date'].dt.strftime('%m-%d').unique()))

무료 개방일: ['01-28']


## 1. 주차별 합 (명·일)

In [3]:
cols = ['기본 (소수)', '기본 (정수)', '① 서비스 수준 0.9 (소수)', '② 무료 개방일 C_u 120만 (소수)', '③ 1명당 1,500명 (소수)']
wk = R.groupby(['site', '주차'])[['기본 (소수)', '기본 (정수)']].sum().unstack('주차').reindex(SITES)
wk.loc['합계'] = wk.sum()
display(wk.round(1))
sm = R.groupby('site')[cols].sum().reindex(SITES); sm.loc['합계'] = sm.sum()
sm.round(1)

기본 (소수)                      기본 (정수)                     
주차           1      2      3      4       1      2      3      4
site                                                            
경복궁       50.2   39.7   40.4   45.2    54.0   43.0   45.0   47.0
창덕궁       14.8   11.1   10.8   11.5    17.0   15.0   14.0   13.0
창경궁        7.8    5.9    5.9    6.3    12.0   10.0    8.0   10.0
덕수궁       29.0   29.2   30.3   32.3    31.0   31.0   32.0   35.0
종묘         5.9    4.0    4.6    5.7     9.0    8.0    8.0    9.0
동구릉        1.0    1.0    1.1    1.3     6.0    6.0    6.0    6.0
광릉         1.0    0.9    0.9    1.0     6.0    6.0    6.0    6.0
홍유릉        0.4    0.3    0.4    0.4     6.0    6.0    6.0    6.0
사릉         0.1    0.1    0.1    0.1     6.0    6.0    6.0    6.0
태릉         0.3    0.3    0.4    0.5     6.0    6.0    6.0    6.0
정릉         0.6    0.7    0.8    0.9     6.0    6.0    6.0    6.0
헌릉         0.2    0.2    0.3    0.3     6.0    6.0    6.0    6.0
선릉         1.8    1.9    2.3    2.7     6.0    6.0    6.0    6.0
의릉         0.8    0.8    0.8    0.9     6.0    6.0    6.0    6.0
영휘원        0.1    0.1    0.2    0.2     6.0    6.0    6.0    6.0
서오릉        2.1    1.8    1.9    2.1     6.0    6.0    6.0    6.0
서삼릉        0.2    0.2    0.2    0.2     6.0    6.0    6.0    6.0
융릉         1.8    1.5    1.6    1.7     6.0    6.0    6.0    6.0
파주삼릉       0.2    0.2    0.2    0.2     6.0    6.0    6.0    6.0
파주장릉       0.1    0.1    0.1    0.1     6.0    6.0    6.0    6.0
김포장릉       1.4    1.4    1.4    1.5     6.0    6.0    6.0    6.0
양주온릉       0.1    0.1    0.1    0.1     6.0    6.0    6.0    6.0
세종대왕유적     2.0    1.8    1.9    2.0     6.0    6.0    6.0    6.0
합계       122.3  103.6  106.5  117.2   231.0  215.0  215.0  222.0

,기본 (소수),기본 (정수),① 서비스 수준 0.9 (소수),② 무료 개방일 C_u 120만 (소수),"③ 1명당 1,500명 (소수)"
site,,,,,
경복궁,175.5,189.0,196.7,177.0,234.0
창덕궁,48.2,59.0,55.3,48.6,64.3
창경궁,25.9,40.0,30.9,26.2,34.6
덕수궁,120.8,129.0,138.8,121.9,161.0
종묘,20.3,34.0,24.7,20.6,27.0
동구릉,4.5,24.0,5.6,4.5,6.0
광릉,3.9,24.0,4.8,3.9,5.2
홍유릉,1.5,24.0,2.0,1.5,2.0
사릉,0.3,24.0,0.4,0.3,0.4


## 2. 일별 정수 권고표 — 주차별

In [4]:
for k in range(1, 5):
    d = R[R['주차'] == k].pivot(index='site', columns='date', values='기본 (정수)').reindex(SITES)
    d.columns = [c.strftime('%m-%d') + '(' + '월화수목금토일'[c.dayofweek] + ')' for c in d.columns]
    d['합'] = d.sum(axis=1); d.loc['합계'] = d.sum()
    print(f'■ {k}주차'); display(d)

■ 1주차


,01-01(목),01-02(금),01-03(토),01-04(일),01-05(월),01-06(화),01-07(수),합
site,,,,,,,,
경복궁,10,9,10,9,8,0,8,54
창덕궁,3,3,3,3,0,3,2,17
창경궁,2,2,2,2,0,2,2,12
덕수궁,5,5,6,5,0,5,5,31
종묘,2,1,2,2,1,0,1,9
동구릉,1,1,1,1,0,1,1,6
광릉,1,1,1,1,0,1,1,6
홍유릉,1,1,1,1,0,1,1,6
사릉,1,1,1,1,0,1,1,6


■ 2주차


,01-08(목),01-09(금),01-10(토),01-11(일),01-12(월),01-13(화),01-14(수),합
site,,,,,,,,
경복궁,7,7,8,7,7,0,7,43
창덕궁,2,2,3,3,0,3,2,15
창경궁,1,1,2,2,0,2,2,10
덕수궁,5,5,6,5,0,5,5,31
종묘,1,1,2,2,1,0,1,8
동구릉,1,1,1,1,0,1,1,6
광릉,1,1,1,1,0,1,1,6
홍유릉,1,1,1,1,0,1,1,6
사릉,1,1,1,1,0,1,1,6


■ 3주차


,01-15(목),01-16(금),01-17(토),01-18(일),01-19(월),01-20(화),01-21(수),합
site,,,,,,,,
경복궁,7,7,8,8,7,0,8,45
창덕궁,2,2,3,2,0,3,2,14
창경궁,1,1,2,2,0,1,1,8
덕수궁,5,5,7,5,0,5,5,32
종묘,1,1,2,2,1,0,1,8
동구릉,1,1,1,1,0,1,1,6
광릉,1,1,1,1,0,1,1,6
홍유릉,1,1,1,1,0,1,1,6
사릉,1,1,1,1,0,1,1,6


■ 4주차


,01-22(목),01-23(금),01-24(토),01-25(일),01-26(월),01-27(화),01-28(수),합
site,,,,,,,,
경복궁,7,7,8,8,7,0,10,47
창덕궁,2,2,2,2,0,3,2,13
창경궁,1,1,2,2,0,2,2,10
덕수궁,5,6,7,5,0,6,6,35
종묘,1,1,2,2,1,0,2,9
동구릉,1,1,1,1,0,1,1,6
광릉,1,1,1,1,0,1,1,6
홍유릉,1,1,1,1,0,1,1,6
사릉,1,1,1,1,0,1,1,6


## 3. 1/28 마지막 수요일(무료) vs 1/21 평소 수요일

모형이 올린 배수는 과거 효과 크기(궁 마지막 수요일 약 1.8배)보다 작다. 백테스트에서 C3 의 마지막 수요일 적중률은 확정 시점별로 0.49(전날) · 0.54(1주 전) · 0.52(2주 전) · 0.61(3주 전)으로, **4주차라서 낮은 것이 아니라** 어느 시점이든 실제가 중앙값의 약 1.5배인 **무료 효과 과소 추정** 때문이다. 1/28 은 권고보다 더 필요할 수 있다.

In [5]:
a = R[R['date'] == '2026-01-28'].set_index('site')['q80']; b = R[R['date'] == '2026-01-21'].set_index('site')['q80']
(a / b).reindex(SITES).round(2).rename('q0.8 배수 (1/28 ÷ 1/21)').to_frame().T

site,경복궁,창덕궁,창경궁,덕수궁,종묘,동구릉,광릉,홍유릉,사릉,태릉,정릉,헌릉,선릉,의릉,영휘원,서오릉,서삼릉,융릉,파주삼릉,파주장릉,김포장릉,양주온릉,세종대왕유적
q0.8 배수 (1/28 ÷ 1/21),1.3,1.14,1.29,1.21,1.9,1.11,1.1,1.08,1.02,1.39,1.25,1.34,1.46,1.25,1.22,1.34,1.14,1.05,1.01,1.12,1.09,1.04,1.08


## 4. 올림과 최소 인원이 만드는 차이

팀 결정(10/15): 날마다 올림 + **왕릉은 문 여는 날 하루 1명 이상**, 관리소 공유 없음. 왕릉은 필요 인원이 하루 0.0~0.4명이라 올림만으로도 하루 1명이 되므로 정수 권고표는 같고, 소수 열(`최종 권고 (소수)`)만 하한 1명이 반영된다.

In [6]:
g = R.groupby('group')[['기본 (소수)', '최종 권고 (소수)', '기본 (정수)']].sum()
g['정수 ÷ 소수'] = g['기본 (정수)'] / g['기본 (소수)']
g.round(2)

,기본 (소수),최종 권고 (소수),기본 (정수),정수 ÷ 소수
group,,,,
궁,370.43,370.43,417,1.13
왕릉,58.90,432.00,432,7.34
종묘,20.27,20.27,34,1.68
